# meinRad Mainz: live bike-sharing snapshots

This notebook visualizes station-level `meinRad` availability snapshots for Mainz. It can read existing CSV snapshots, collect new snapshots, and create interactive maps and time animations when several snapshots are available.

**Learning goals**

- Load repeated live API snapshots as a time-dependent geospatial dataset.
- Map current station availability and floating bikes.
- Animate changes over time with a time slider.
- Identify empty, full, and highly variable stations.

In [ ]:
# Colab/Jupyter setup
!pip -q install pandas plotly

In [ ]:
from pathlib import Path
import subprocess
import sys
import time

import pandas as pd
import plotly.express as px
pd.set_option("display.max_columns", 80)

## 1. Paths and optional live collection

If you open this notebook in Colab from GitHub, first clone the repository or upload the data folder. Locally, run it from the repository root.

In [ ]:
ROOT = Path.cwd()
if not (ROOT / "data_loaders" / "meinrad").exists():
    # Common Colab location after: !git clone https://github.com/yfeng-hsm/KI_Geodatenanalyse_SS26.git
    candidate = Path("/content/KI_Geodatenanalyse_SS26")
    if (candidate / "data_loaders" / "meinrad").exists():
        ROOT = candidate

DATA_DIR = ROOT / "data_loaders" / "meinrad" / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Repository root:", ROOT)
print("Snapshot folder:", DATA_DIR)

In [ ]:
# Set this to True when you want the notebook to collect new live snapshots.
# For a real temporal animation, collect every 5-15 minutes over several hours or days.
COLLECT_NEW_SNAPSHOTS = False
N_SNAPSHOTS = 3
WAIT_SECONDS = 300

if COLLECT_NEW_SNAPSHOTS:
    for i in range(N_SNAPSHOTS):
        print(f"Collecting snapshot {i + 1}/{N_SNAPSHOTS} ...")
        subprocess.run(
            [sys.executable, "-m", "data_loaders.meinrad.src.meinrad_snapshot"],
            cwd=ROOT,
            check=True,
        )
        if i < N_SNAPSHOTS - 1:
            time.sleep(WAIT_SECONDS)

## 2. Load all station snapshots

In [ ]:
snapshot_files = sorted(DATA_DIR.glob("meinrad_mainz_places_*.csv"))
if not snapshot_files:
    raise FileNotFoundError(
        f"No snapshot CSV files found in {DATA_DIR}. Run the collector first."
    )

df = pd.concat((pd.read_csv(path) for path in snapshot_files), ignore_index=True)
df["collected_at_utc"] = pd.to_datetime(df["collected_at_utc"], utc=True)
df["snapshot_label"] = df["collected_at_utc"].dt.strftime("%Y-%m-%d %H:%M UTC")

numeric_cols = [
    "lat",
    "lng",
    "bikes_available_to_rent",
    "bikes",
    "bike_racks",
    "free_racks",
    "bike_count_from_list",
]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

stations = df[df["is_station"] == True].copy()
floating = df[df["is_floating_bike"] == True].copy()

print(f"Loaded {len(snapshot_files)} snapshot file(s)")
print(f"Rows: {len(df):,}; stations: {len(stations):,}; floating-bike rows: {len(floating):,}")
print("Snapshot times:")
display(df[["collected_at_utc", "snapshot_label"]].drop_duplicates().sort_values("collected_at_utc"))

## 3. Current station availability map

The latest snapshot is shown first. Marker size represents available bikes; color represents station fill ratio.

In [ ]:
latest_time = df["collected_at_utc"].max()
latest_stations = stations[stations["collected_at_utc"] == latest_time].copy()
latest_stations["fill_ratio"] = (
    latest_stations["bikes_available_to_rent"]
    / latest_stations["bike_racks"].where(latest_stations["bike_racks"] > 0)
)
latest_stations["marker_size"] = latest_stations["bikes_available_to_rent"].clip(lower=1)

fig_current = px.scatter_mapbox(
    latest_stations,
    lat="lat",
    lon="lng",
    size="marker_size",
    color="fill_ratio",
    color_continuous_scale="Viridis",
    range_color=(0, 1),
    hover_name="name",
    hover_data={
        "bikes_available_to_rent": True,
        "bike_racks": True,
        "free_racks": True,
        "maintenance": True,
        "lat": False,
        "lng": False,
        "marker_size": False,
        "fill_ratio": ":.2f",
    },
    zoom=11,
    height=650,
    title=f"meinRad station availability, {latest_time:%Y-%m-%d %H:%M UTC}",
)
fig_current.update_layout(mapbox_style="open-street-map", margin={"r": 0, "t": 45, "l": 0, "b": 0})
fig_current.show()

## 4. Dynamic map over time

This cell becomes a real animation after several snapshots have been collected. Use the play button or the time slider.

In [ ]:
snapshot_count = stations["collected_at_utc"].nunique()

if snapshot_count < 2:
    print(
        "Only one snapshot is available. Set COLLECT_NEW_SNAPSHOTS=True above, "
        "or run the collector repeatedly, then rerun this cell for animation."
    )
else:
    anim = stations.copy()
    anim["marker_size"] = anim["bikes_available_to_rent"].clip(lower=1)
    fig_anim = px.scatter_mapbox(
        anim,
        lat="lat",
        lon="lng",
        animation_frame="snapshot_label",
        animation_group="place_uid",
        size="marker_size",
        color="bikes_available_to_rent",
        color_continuous_scale="Turbo",
        hover_name="name",
        hover_data={
            "bikes_available_to_rent": True,
            "bike_racks": True,
            "free_racks": True,
            "maintenance": True,
            "lat": False,
            "lng": False,
            "marker_size": False,
        },
        zoom=11,
        height=650,
        title="meinRad station availability over time",
    )
    fig_anim.update_layout(mapbox_style="open-street-map", margin={"r": 0, "t": 45, "l": 0, "b": 0})
    fig_anim.show()

## 5. Station imbalance and change

In [ ]:
latest_view = latest_stations[
    ["name", "bikes_available_to_rent", "bike_racks", "free_racks", "maintenance"]
].sort_values("bikes_available_to_rent")

display(latest_view.head(15).rename(columns={"bikes_available_to_rent": "available_bikes"}))

top_available = latest_view.sort_values("bikes_available_to_rent", ascending=False).head(15)
fig_bar = px.bar(
    top_available.sort_values("bikes_available_to_rent"),
    x="bikes_available_to_rent",
    y="name",
    orientation="h",
    title="Stations with the most available bikes in the latest snapshot",
    labels={"bikes_available_to_rent": "Available bikes", "name": "Station"},
    height=500,
)
fig_bar.show()

In [ ]:
if snapshot_count < 2:
    print("Change metrics need at least two snapshots.")
else:
    station_variation = (
        stations.groupby(["place_uid", "name"], as_index=False)
        .agg(
            min_available=("bikes_available_to_rent", "min"),
            max_available=("bikes_available_to_rent", "max"),
            mean_available=("bikes_available_to_rent", "mean"),
            snapshots=("collected_at_utc", "nunique"),
        )
    )
    station_variation["range_available"] = (
        station_variation["max_available"] - station_variation["min_available"]
    )
    display(station_variation.sort_values("range_available", ascending=False).head(20))

    top_dynamic_ids = station_variation.sort_values("range_available", ascending=False).head(12)["place_uid"]
    top_dynamic = stations[stations["place_uid"].isin(top_dynamic_ids)]
    fig_lines = px.line(
        top_dynamic,
        x="collected_at_utc",
        y="bikes_available_to_rent",
        color="name",
        markers=True,
        title="Most variable stations across collected snapshots",
        labels={"collected_at_utc": "Time", "bikes_available_to_rent": "Available bikes"},
        height=550,
    )
    fig_lines.show()

## 6. Export an interactive HTML map

In [ ]:
EXPORT_HTML = False

if EXPORT_HTML:
    output_path = DATA_DIR / "meinrad_mainz_latest_map.html"
    fig_current.write_html(output_path)
    print("Wrote", output_path)